# StudyFlow — Exploratory Data Analysis (EDA)
## Academic Performance, Workload Stress & Task Completion Modeling

**Author:** StudyFlow Data Science Team  
**Dataset:** `academic_study_dataset.csv` (3,500 records)  
**Objective:** End-to-end exploratory analysis of academic features predicting on-time completion probability, true execution duration, and course vulnerability risk.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style='whitegrid', font='sans-serif')
%matplotlib inline

csv_path = 'academic_study_dataset.csv'
df = pd.read_csv(csv_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 1. Data Integrity: Missing Values & Duplicates
Verifying data completeness before statistical modeling.

In [2]:
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

### 2. Summary Statistics
Descriptive metrics for continuous and ordinal features.

In [3]:
numerical_cols = [
    'subject_difficulty', 'priority_code', 'estimated_duration', 'days_until_due',
    'subtask_count', 'student_study_streak', 'hist_completion_rate',
    'past_studied_minutes', 'class_hours_today', 'completed_on_time', 'actual_duration'
]
df[numerical_cols].describe().T

### 3. Target Variable Distributions
Examining class balance (`completed_on_time`) and actual duration distribution (`actual_duration`).

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Classification Target
counts = df['completed_on_time'].value_counts()
ax1.pie(counts, labels=['Late / Incomplete (0)', 'Completed On Time (1)'], autopct='%1.1f%%', colors=['#ef4444', '#10b981'], startangle=140, explode=(0.04, 0))
ax1.set_title("Classification Target Balance: completed_on_time", fontsize=12, fontweight='bold')

# Regression Target
sns.histplot(df['actual_duration'], kde=True, ax=ax2, color='#6366f1')
ax2.axvline(df['actual_duration'].mean(), color='red', linestyle='--', label=f'Mean: {df["actual_duration"].mean():.1f}m')
ax2.set_title("Regression Target: actual_duration (minutes)", fontsize=12, fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()

### 4. Correlation Analysis
Analyzing pairwise relationships between academic workload signals.

In [5]:
corr = df[numerical_cols].corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, mask=mask)
plt.title("Correlation Matrix of Academic Workload Features", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 5. Feature-Target Relationships
How do deadline proximity, past preparation, and subject difficulty impact completion?

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x='days_until_due', y='completed_on_time', ax=axes[0], color='#4f46e5')
axes[0].set_title("Completion Probability by Days Until Deadline", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Completion Rate")

sns.barplot(data=df, x='subject', y='completed_on_time', ax=axes[1], palette='Set2')
axes[1].set_title("Completion Rate by Academic Subject Difficulty", fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=25)
axes[1].set_ylabel("Completion Rate")

plt.tight_layout()
plt.show()

### 6. Planning Fallacy Analysis
Examining student time underestimation (Estimated vs Actual Duration).

In [7]:
plt.figure(figsize=(8, 6))
sample_df = df.sample(400, random_state=42)
sns.scatterplot(data=sample_df, x='estimated_duration', y='actual_duration', hue='subject', alpha=0.7)
plt.plot([0, 300], [0, 300], '--', color='red', label='1:1 Parity')
plt.title("Student Planning Fallacy: Actual vs Estimated Duration", fontsize=12, fontweight='bold')
plt.xlabel("Estimated Duration (minutes)")
plt.ylabel("Actual Logged Duration (minutes)")
plt.legend()
plt.tight_layout()
plt.show()